# Erhebung Korpus 1 — Schweizer Legal-Tech-Marketingtexte

Setzt `Eymann_Workflow_Korpus1.md` Schritt 1 um. Ziel: Marketingtexte der 29
Kern-Anbieter aus `Eymann_Anbieterliste_Korpus1.md` erheben.

Dieses Notebook ist die kanonische Quelle für `Eymann_Korpus1_roh.csv`: ein
vollständiger Durchlauf (Erhebungsschleife + Nacherhebungen) erzeugt bzw.
aktualisiert die Datei direkt. Reine Beobachtungen zur Erhebung (Redirects,
auffällig kurzer Text etc.) werden automatisch erkannt und in der Spalte
`anmerkung_erhebung` abgelegt — nicht im `rohtext` selbst, damit dieser
ausschliesslich reinen Korpustext enthält.

In [ ]:
# Imports
import time
import urllib.robotparser as robotparser
from urllib.parse import urlparse
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import trafilatura  # pip install trafilatura — siehe requirements.txt
import pandas as pd

PFAD_ROH = "../daten/korpus1/Eymann_Korpus1_roh.csv"
KURZTEXT_SCHWELLE = 100  # Wörter; darunter wird ein Hinweis vermerkt


## Anbieterliste (29 Kern-Anbieter aus `Eymann_Anbieterliste_Korpus1.md`)

In [ ]:
anbieter_urls = [
    ("DeepJudge", "https://www.deepjudge.ai/"),
    ("iuslex.ch (IUS)", "https://iuslex.ch/"),
    ("DeepLegal", "https://www.deeplegal.swiss/"),
    ("Justement", "https://justement.ch/de"),
    ("Legartis", "https://www.legartis.ai/"),
    ("Omnilex", "https://omnilex.ai/en/"),
    ("Herlock.ai", "https://www.herlock.ai/"),
    ("Lawise.ai (Jurilo)", "https://lawise.ai/"),
    ("Bryter", "https://bryter.com/"),
    ("Swisslex", "https://www.swisslex.ch/"),
    ("CASUS", "https://casus.ch/"),
    ("Leya", "https://leya.law/"),
    ("balo.ai", "https://balo.ai/"),
    ("ExNunc Intelligence", "https://exnuncintelligence.com/"),
    ("Laine", "https://laine.ai/"),
    ("whisperit", "https://whisperit.ai/"),
    ("Contractus Intelligence", "https://contractus.ai/"),
    ("IPQuants", "https://ipquants.com/"),
    ("Weblaw/LegalTechHub", "https://legaltech.weblaw.ch/en/legaltech.html"),
    ("BetterCallClaude", "https://bettercallclaude.ch/"),
    ("Abacus Law / AbaLaw", "https://www.abacus.ch/en/law"),
    ("DocIQ", "https://dociq.io/"),
    ("Lexplorer", "https://lexplorer.ch/"),
    ("Libra (Wolters Kluwer)", "https://libratech.ai/"),
    ("Silex", "https://silex.legal/"),
    ("Jurata", "https://www.jurata.ch/en/"),
    ("Lawcodex", "https://lawcodex.ch/"),
    ("Elisa / legalis", "https://www.legalis.ch/de/elisa/"),
    ("Swiss-Noxtua", "https://swiss-noxtua.ch/"),
]
print(f"{len(anbieter_urls)} Anbieter geladen")


## Hilfsfunktionen: robots.txt-Check, Textextraktion, automatische Anmerkungen, CSV-Update

"Gently" scrapen heisst hier: robots.txt vor jedem Abruf prüfen (Sperren
respektieren), einen expliziten Crawl-Delay aus der robots.txt übernehmen
falls vorhanden (sonst konservativer Default), uns über den User-Agent
identifizierbar machen (mit Kontakt-E-Mail), sequenziell statt parallel
abrufen, und nur je eine Seite pro Anbieter — kein aggressives Crawling.

In [ ]:
DEFAULT_DELAY = 2.0  # Sekunden, falls robots.txt keinen Crawl-Delay nennt

def robots_check(url, user_agent="EymannResearchBot"):
    """Prüft robots.txt: (erlaubt: bool, crawl_delay: float)."""
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = robotparser.RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        erlaubt = rp.can_fetch(user_agent, url)
        delay = rp.crawl_delay(user_agent)
        return erlaubt, float(delay) if delay else DEFAULT_DELAY
    except Exception:
        # Keine robots.txt auffindbar/lesbar -> als nicht gesperrt werten,
        # trotzdem konservativen Default-Delay verwenden
        return True, DEFAULT_DELAY


def haupttext_extrahieren(html, url=""):
    """Extrahiert Haupttext via trafilatura (bevorzugt, weniger Navigations-/
    Menü-Rauschen), Fallback/Vergleich mit BeautifulSoup.

    trafilatura ist auf normales Server-HTML trainiert. Bei per DevTools aus
    einem echten Browser kopiertem, bereits gerendertem DOM (viele Framework-
    Wrapper-Divs, Hydration-Markup) unterschätzt es die Haupttextmenge teils
    massiv, liefert aber genug Zeichen, um nicht als 'leer' zu gelten. Deshalb:
    beide Extraktionen vergleichen und die deutlich längere nehmen, statt
    trafilatura blind zu vertrauen, sobald es die Mindestlänge überschreitet."""
    text_trafilatura = trafilatura.extract(html, url=url, include_comments=False, include_tables=False) or ""

    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()
    text_bs4 = soup.get_text(separator=" ", strip=True)

    if len(text_bs4) > len(text_trafilatura) * 1.5 and len(text_bs4) > 100:
        return text_bs4
    return text_trafilatura if len(text_trafilatura) > 50 else text_bs4


def _netloc(url):
    return urlparse(url).netloc.replace("www.", "")


BEKANNTE_ANMERKUNGEN = {
    # Spezialwissen, das sich nicht rein aus HTTP-Verhalten ableiten lässt und
    # daher bei jedem Rerun explizit verankert werden muss (sonst geht es beim
    # nächsten vollständigen Durchlauf wieder verloren).
    "ExNunc Intelligence": (
        "Domain ist eine Hosting-Platzhalterseite ohne Marketinginhalt; das "
        "operative Produkt (Ex Nunc Intelligence SA) wird separat unter "
        "silex.legal vermarktet (siehe Eintrag 'Silex')."
    ),
}


def anmerkung_automatisch(url_angefragt, url_effektiv, rohtext, anbieter=None):
    """Erkennt automatisch zwei Fälle, die für die spätere Textanalyse relevant
    sind, aber NICHT im rohtext selbst landen dürfen: (a) eine Weiterleitung auf
    eine andere Domain (Rebranding/Konsolidierung), (b) auffällig kurzer Text
    (z.B. weil die Seite primär als UI/App statt als Fliesstext funktioniert).
    Ergänzt zusätzlich fest hinterlegtes Spezialwissen (BEKANNTE_ANMERKUNGEN),
    falls für den Anbieter vorhanden."""
    hinweise = []
    if url_effektiv and _netloc(url_angefragt) != _netloc(url_effektiv):
        hinweise.append(
            f"Weiterleitung von {_netloc(url_angefragt)} auf {_netloc(url_effektiv)} "
            "(Rebranding/Konsolidierung? manuell prüfen)."
        )
    wortanzahl = len(rohtext.split()) if rohtext else 0
    if 0 < wortanzahl < KURZTEXT_SCHWELLE:
        hinweise.append(
            f"Kurzer Text ({wortanzahl} Wörter) – ggf. UI-/App-lastige oder "
            "dynamisch gerenderte Seite, manuell prüfen."
        )
    if anbieter in BEKANNTE_ANMERKUNGEN:
        hinweise.append(BEKANNTE_ANMERKUNGEN[anbieter])
    return " ".join(hinweise)


def in_rohdaten_einspielen(df_updates, methode, pfad=PFAD_ROH):
    """Spielt erfolgreiche (Nach-)Erhebungen (Spalten: anbieter, rohtext, status,
    optional anmerkung_erhebung) anhand des Anbieter-Namens in die zentrale CSV
    ein und dokumentiert die Erhebungsmethode
    (Spalte 'erhebungsmethode': automatisiert / manuell_browser)."""
    df_original = pd.read_csv(pfad) if Path(pfad).exists() else pd.DataFrame()
    for spalte, default in [("erhebungsmethode", "automatisiert"), ("anmerkung_erhebung", "")]:
        if spalte not in df_original.columns:
            df_original[spalte] = default
    for _, zeile in df_updates.iterrows():
        maske = df_original["anbieter"] == zeile["anbieter"]
        if zeile["status"] == "ok":
            df_original.loc[maske, "rohtext"] = zeile["rohtext"]
            df_original.loc[maske, "status"] = "ok"
            df_original.loc[maske, "datum_erhoben"] = pd.Timestamp.today().strftime("%Y-%m-%d")
            df_original.loc[maske, "erhebungsmethode"] = methode
            df_original.loc[maske, "anmerkung_erhebung"] = zeile.get("anmerkung_erhebung", "")
    df_original.to_csv(pfad, index=False)
    print("Aktualisiert. Neue Status-Verteilung:")
    print(df_original["status"].value_counts())
    return df_original


## Erhebungsschleife

Erzeugt `Eymann_Korpus1_roh.csv` direkt neu (alle 29 Anbieter, Methode
`automatisiert`). Nacherhebungen (Playwright, manueller Browser-Snapshot)
aktualisieren anschliessend gezielt nur die dort weiterhin fehlgeschlagenen
Zeilen.

In [ ]:
headers = {"User-Agent": "EymannResearchBot/1.0 (Masterseminararbeit LUMACSS, Kontakt: b.eymann@bluewin.ch)"}

rows = []
for i, (anbieter, url) in enumerate(anbieter_urls, start=1):
    status = "ok"
    rohtext = ""
    anmerkung = ""
    erlaubt, delay = robots_check(url)
    try:
        if not erlaubt:
            status = "fehler: robots.txt sperrt /"
        else:
            resp = requests.get(url, headers=headers, timeout=15, allow_redirects=True)
            if resp.status_code != 200:
                status = f"fehler: HTTP {resp.status_code}"
            else:
                # requests rät die Kodierung mangels Server-Angabe manchmal falsch
                # (Standard-Fallback ISO-8859-1) -> führt zu kaputten Umlauten/Akzenten.
                # apparent_encoding (chardet) ist dafür zuverlässiger.
                if not resp.encoding or resp.encoding.lower() == "iso-8859-1":
                    resp.encoding = resp.apparent_encoding
                rohtext = haupttext_extrahieren(resp.text, resp.url)
                if not rohtext or len(rohtext) < 50:
                    status = "fehler: leer/JS-basiert, nicht auslesbar"
                else:
                    anmerkung = anmerkung_automatisch(url, resp.url, rohtext, anbieter=anbieter)
    except Exception as e:
        status = f"fehler: {type(e).__name__}"

    rows.append({
        "id": i,
        "anbieter": anbieter,
        "url": url,
        "seitentyp": "startseite",
        "datum_erhoben": pd.Timestamp.today().strftime("%Y-%m-%d"),
        "rohtext": rohtext,
        "status": status,
        "erhebungsmethode": "automatisiert",
        "anmerkung_erhebung": anmerkung,
    })
    time.sleep(delay)  # Rate-Limiting: robots.txt-Crawl-Delay falls vorhanden, sonst Default

df_neu = pd.DataFrame(rows)
df_neu.to_csv(PFAD_ROH, index=False)
print(df_neu["status"].value_counts())


## Analyse des Rohdatensatzes

In [ ]:
df = pd.read_csv(PFAD_ROH)
df["wortanzahl"] = df["rohtext"].fillna("").apply(lambda t: len(t.split()))
df[["anbieter", "status", "erhebungsmethode", "anmerkung_erhebung", "wortanzahl"]]


In [ ]:
print("Status-Verteilung:")
print(df["status"].value_counts())
print()
print("Wortanzahl (nur erfolgreiche):")
print(df.loc[df["status"] == "ok", "wortanzahl"].describe())


## Nacherhebung JS-basierter und dünner Seiten (Playwright)

Manche Seiten laden ihren Inhalt erst per JavaScript nach — reines
`requests` + `trafilatura` liefert dann nur ein leeres HTML-Grundgerüst oder
nur einen Bruchteil des tatsächlichen Texts (z.B. FAQ/Preise/Testimonials,
die erst nach dem Laden per JS eingefügt werden — sichtbar an der Anmerkung
"Kurzer Text"). Für beide Fälle rendert Playwright die Seite zuerst mit einem
echten (headless) Chromium-Browser.

**Einmalige Installation (Terminal, nicht im Notebook):**
```bash
pip install playwright
playwright install chromium
```

In [ ]:
from playwright.sync_api import sync_playwright

def rendern_und_extrahieren(url, timeout_ms=20000):
    """Rendert die Seite mit einem headless Chromium-Browser und extrahiert
    danach den Haupttext wie gewohnt via trafilatura. Gibt zusätzlich die
    effektive URL nach etwaigen Weiterleitungen zurück."""
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page(user_agent="EymannResearchBot/1.0 (Masterseminararbeit LUMACSS)")
        page.goto(url, timeout=timeout_ms, wait_until="networkidle")
        html = page.content()
        url_effektiv = page.url
        browser.close()
    return haupttext_extrahieren(html, url_effektiv), url_effektiv


In [ ]:
JS_SPA_ANBIETER = ["iuslex.ch (IUS)", "Justement", "Swisslex", "BetterCallClaude", "Lawcodex"]

nachzuholen = [(a, u) for a, u in anbieter_urls if a in JS_SPA_ANBIETER]

nachgeholt = []
for anbieter, url in nachzuholen:
    erlaubt, delay = robots_check(url)
    text, anmerkung = "", ""
    try:
        if not erlaubt:
            status = "fehler: robots.txt sperrt /"
        else:
            text, url_effektiv = rendern_und_extrahieren(url)
            if text and len(text) > 50:
                status = "ok"
                anmerkung = anmerkung_automatisch(url, url_effektiv, text, anbieter=anbieter)
            else:
                status = "fehler: weiterhin leer nach Rendering"
    except Exception as e:
        status = f"fehler: {type(e).__name__}"
    nachgeholt.append({"anbieter": anbieter, "rohtext": text, "status": status, "anmerkung_erhebung": anmerkung})
    print(anbieter, "->", status, f"({len(text)} Zeichen)" if text else "")
    time.sleep(delay)

df_nachgeholt = pd.DataFrame(nachgeholt)
in_rohdaten_einspielen(df_nachgeholt, methode="automatisiert")


## Manuelle Nacherhebung via Browser-Snapshot (falls Playwright weiterhin scheitert)

Manche Seiten erkennen und blockieren Headless-Browser (Bot-Erkennung), auch
wenn `requests` und Playwright beide scheitern. Der zuverlässigste Ausweg:
ein echter, sichtbarer Browser, den du selbst bedienst — Cookie-Banner
wegklicken inklusive. Das gerenderte HTML holen wir uns danach einmalig ab
und verarbeiten es wie gewohnt weiter.

**Pro betroffener Seite — im Browser, nicht im Notebook:**

1. Seite in deinem normalen Chrome/Edge/Firefox öffnen, warten bis sie
   vollständig geladen ist.
2. Cookie-Banner/Consent-Dialog wegklicken, falls einer erscheint.
3. Entwicklertools öffnen: Taste **F12** (oder Rechtsklick → "Untersuchen").
4. Zum Tab **Console** wechseln.
5. Dort eintippen und Enter drücken:
   ```js
   copy(document.documentElement.outerHTML)
   ```
   Das kopiert das komplette, fertig gerenderte HTML in die Zwischenablage.
6. Editor öffnen (z.B. Notepad), **Strg+V** einfügen, als UTF-8-Textdatei
   speichern unter (Dateiname jeweils passend):
   ```
   daten/korpus1/manuell/iuslex.html
   daten/korpus1/manuell/justement.html
   daten/korpus1/manuell/swisslex.html
   daten/korpus1/manuell/bettercallclaude.html
   daten/korpus1/manuell/lawcodex.html
   ```

Wenn alle 5 Dateien gespeichert sind, weiter mit der Zelle unten.

In [ ]:
manuell_dateien = {
    "iuslex.ch (IUS)": "iuslex.html",
    "Justement": "justement.html",
    "Swisslex": "swisslex.html",
    "BetterCallClaude": "bettercallclaude.html",
    "Lawcodex": "lawcodex.html",
}

manuell_ordner = Path("../daten/korpus1/manuell")
nachgeholt_manuell = []
for anbieter, dateiname in manuell_dateien.items():
    pfad_datei = manuell_ordner / dateiname
    if not pfad_datei.exists():
        print(f"{anbieter}: Datei {pfad_datei} fehlt noch — übersprungen.")
        continue
    html = pfad_datei.read_text(encoding="utf-8")
    text = haupttext_extrahieren(html)
    status = "ok" if text and len(text) > 50 else "fehler: auch im Snapshot leer"
    nachgeholt_manuell.append({"anbieter": anbieter, "rohtext": text, "status": status})
    print(anbieter, "->", status, f"({len(text)} Zeichen)")

df_nachgeholt_manuell = pd.DataFrame(nachgeholt_manuell)
in_rohdaten_einspielen(df_nachgeholt_manuell, methode="manuell_browser")


## Fälle mit "Kurzer Text" gezielt nachholen (Playwright)

Manche automatisiert erhobenen Seiten laden Teile ihres Inhalts (FAQ, Preise,
Testimonials) erst per JavaScript nach — reines `requests` bekommt davon
nichts mit und der Text bleibt auffällig kurz (Anmerkung "Kurzer Text").
Playwright rendert hier nach; übernommen wird das Ergebnis nur, wenn es
tatsächlich *mehr* Text liefert als die bisherige Version — sonst war die
Seite schlicht kurz, und wir behalten die bisherige, bereits geprüfte
Version. Die 5 bereits per Browser-Snapshot gelösten JS-SPA-Fälle werden
hier ausgeklammert.

In [ ]:
df = pd.read_csv(PFAD_ROH)  # aktuellen Stand nach allen bisherigen Nacherhebungen laden
url_lookup = dict(anbieter_urls)

# Aktuell bekannte "Kurzer Text"-Fälle (Stand 22.07.2026):
KURZTEXT_KANDIDATEN = ["DeepLegal", "balo.ai", "ExNunc Intelligence", "Libra (Wolters Kluwer)"]

# Gegenprobe: stimmen die hinterlegten Kandidaten noch mit der aktuellen CSV überein?
tatsaechlich_kurz = set(df.loc[
    df["anmerkung_erhebung"].fillna("").str.contains("Kurzer Text")
    & ~df["anbieter"].isin(JS_SPA_ANBIETER),
    "anbieter",
])
if tatsaechlich_kurz != set(KURZTEXT_KANDIDATEN):
    print("Achtung: KURZTEXT_KANDIDATEN weicht von der aktuellen CSV ab.")
    print("  Nur in Liste:", set(KURZTEXT_KANDIDATEN) - tatsaechlich_kurz)
    print("  Nur in CSV:  ", tatsaechlich_kurz - set(KURZTEXT_KANDIDATEN))

kandidaten_duenn = KURZTEXT_KANDIDATEN
print("Kandidaten für Nacherhebung:", kandidaten_duenn)

nachgeholt_duenn = []
for anbieter in kandidaten_duenn:
    url = url_lookup[anbieter]
    alte_wortanzahl = len(df.loc[df["anbieter"] == anbieter, "rohtext"].iloc[0].split())
    erlaubt, delay = robots_check(url)
    text, anmerkung, status = "", "", ""
    try:
        if not erlaubt:
            status = "fehler: robots.txt sperrt /"
        else:
            text, url_effektiv = rendern_und_extrahieren(url)
            neue_wortanzahl = len(text.split()) if text else 0
            if neue_wortanzahl > alte_wortanzahl:
                status = "ok"
                anmerkung = anmerkung_automatisch(url, url_effektiv, text, anbieter=anbieter)
            else:
                status = f"fehler: Playwright brachte keine längere Version ({neue_wortanzahl} <= {alte_wortanzahl} Wörter)"
                text = ""
    except Exception as e:
        status = f"fehler: {type(e).__name__}"
    nachgeholt_duenn.append({"anbieter": anbieter, "rohtext": text, "status": status, "anmerkung_erhebung": anmerkung})
    print(anbieter, "->", status)
    time.sleep(delay)

df_nachgeholt_duenn = pd.DataFrame(nachgeholt_duenn)
in_rohdaten_einspielen(df_nachgeholt_duenn, methode="automatisiert")


## Falls Playwright bei den "Kurzer Text"-Fällen ebenfalls nichts bringt: manueller Browser-Snapshot

Playwright rendert zwar JavaScript, löst aber nicht jeden Fall (z.B. Inhalte,
die erst nach Scrollen/Klicken nachladen, oder erneute Bot-Erkennung). Gleicher
Ausweg wie bei den JS-SPA-Fällen weiter oben: Seite in einem echten, sichtbaren
Browser öffnen und das gerenderte HTML manuell sichern.

**ExNunc Intelligence ausgenommen:** Das ist nachweislich eine echte
Platzhalterseite (kein Marketinginhalt vorhanden) — hier bringt auch ein
manueller Snapshot nichts.

**Pro betroffener Seite (DeepLegal, balo.ai, Libra) — im Browser, nicht im Notebook:**

1. Seite öffnen, warten bis vollständig geladen; ggf. bewusst runterscrollen,
   falls Inhalt lazy-loaded wird.
2. Cookie-Banner wegklicken, falls vorhanden.
3. **F12** → Tab **Console** → eintippen und Enter:
   ```js
   copy(document.documentElement.outerHTML)
   ```
4. In Notepad einfügen, als UTF-8-Textdatei speichern unter:
   ```
   daten/korpus1/manuell/deeplegal.html
   daten/korpus1/manuell/baloai.html
   daten/korpus1/manuell/libra.html
   ```

URLs: DeepLegal → https://www.deeplegal.swiss/, balo.ai → https://balo.ai/,
Libra → https://libratech.ai/

Wenn die Dateien gespeichert sind, weiter mit der Zelle unten. Übernommen wird
auch hier nur, was tatsächlich mehr Text liefert als die bisherige Version.

In [ ]:
manuell_dateien_duenn = {
    "DeepLegal": "deeplegal.html",
    "balo.ai": "baloai.html",
    "Libra (Wolters Kluwer)": "libra.html",
}

df = pd.read_csv(PFAD_ROH)
nachgeholt_manuell_duenn = []
for anbieter, dateiname in manuell_dateien_duenn.items():
    pfad_datei = manuell_ordner / dateiname
    if not pfad_datei.exists():
        print(f"{anbieter}: Datei {pfad_datei} fehlt noch — übersprungen.")
        continue
    alte_wortanzahl = len(df.loc[df["anbieter"] == anbieter, "rohtext"].iloc[0].split())
    html = pfad_datei.read_text(encoding="utf-8")
    text = haupttext_extrahieren(html)
    neue_wortanzahl = len(text.split()) if text else 0
    if neue_wortanzahl > alte_wortanzahl:
        status = "ok"
    else:
        status = f"fehler: Snapshot brachte keine längere Version ({neue_wortanzahl} <= {alte_wortanzahl} Wörter)"
        text = ""
    nachgeholt_manuell_duenn.append({"anbieter": anbieter, "rohtext": text, "status": status})
    print(anbieter, "->", status)

df_nachgeholt_manuell_duenn = pd.DataFrame(nachgeholt_manuell_duenn)
in_rohdaten_einspielen(df_nachgeholt_manuell_duenn, methode="manuell_browser")


## Nächste Schritte

1. Falls einzelne Seiten auch im Browser-Snapshot leer/kurz bleiben: als
   endgültig nicht erhebbar dokumentieren statt weiter zu forcieren.
2. Pro Anbieter zusätzliche Seitentypen erheben (Blog, Produktseiten, Whitepaper)
   — aktuell nur `startseite`.
3. Weiter mit `Eymann_Workflow_Korpus1.md` Schritt 2 (Qualitätskontrolle) und
   Schritt 3 (Bereinigung: Deduplizierung, Spracherkennung, Segmentierung).